# 16 · Distillation, a leaky ReLU, and a loss that weights edges

Ways to get more out of a model without touching what its graph costs in the browser:

- **Distillation.** The cheap tier is the luma 32-16 body (notebook 4). It is trained here also against notebook 8's
  told model: a picture a 32-16 can come closer to than the truth, made by a model told the bitrate and the line,
  which the student is not. Half the loss is L1 against the teacher's picture, the rest the shared loss.
- **A leaky ReLU.** At 1e-3 the 32-16 can switch channels of its narrow layer off for good: a ReLU passes no gradient
  to a channel that is negative everywhere. A leaky ReLU (slope 0.1, `LeakyRelu`, inside the browser's fast set)
  cannot, at the same cost.
- **A loss that weights edges.** `metrics.ssim_l1_loss`'s L1 counts a pixel of flat background like a pixel of a
  glyph's edge, and on a screen the edges are the picture. Here the L1 is weighted by how strong an edge the truth
  has at each pixel, on notebook 8's told model.

Every run is `training.fit` from the start until it converges; the seed sets the starting weights and the batches, so
two runs at one seed differ only in what is being tested. Scored on notebook 0's validation split.

Needs: `00_data_preprocess.ipynb`, the runs of notebook 4 (the luma 32-16 at seed 0) and notebook 8 (the told model,
the teacher), and the scores of notebooks 1, 4 and 8.

In [1]:
"""Settings."""

import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import metrics, models, patches, training, webexport
from helpers.training import DATA_DIR, DEVICE

NOTEBOOK = 16
STUDENT = dict(widths=(32, 16), blocks=0, stem=5)   # notebook 4's luma 32-16
TOLD = ("bpp", "deblocked")                         # what notebook 8's told model, the teacher, is told
COLUMNS = patches.columns(TOLD)
TEACHER_RUN = training.CHECKPOINT_DIR / "told_runs" / "told_bpp_deblocked_seed0.pt"
DISTILL_WEIGHT = 0.5    # the share of the student's loss against the teacher; the rest keeps the SSIM term
LEAKY_SLOPE = 0.1       # a negative value passes at a tenth
EDGE_STRENGTH = 4.0     # an average edge pixel weighs 1 + this in the L1, a flat one 1 (before scaling to mean 1)
RUN_DIR = training.CHECKPOINT_DIR / "distill_runs"

manifest = training.load_manifest()
SCALE = manifest["scale"]
trim = training.trimmer(manifest)   # cuts the halo off an output: the border is padding, not picture
recorded = training.recorded()
bicubic = recorded[1, "bicubic"]    # notebook 1's scores of the bar

## 1. Models and losses

**The teacher** is notebook 8's told model at its best epoch, frozen. **The student** is not told, so `Blind` drops
the condition the told loaders hand it. The edge-weighted loss: `ssim_l1_loss` with its L1 weighted per pixel, 1 on a
flat background and 1 + `EDGE_STRENGTH` at an edge as strong as the training set's average (a Sobel of the truth's
luma), scaled to average 1 over a batch, so the L1 keeps its share against the SSIM term and only moves where it is
spent.

In [2]:
class Blind(nn.Module):
    """A model that is not told, run on loaders that tell: it drops the condition."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, condition=None):
        return self.model(x)


def student(slope=0.0):
    return models.LumaUpscaler(**STUDENT, scale=SCALE, slope=slope).to(DEVICE)


def told(slope=0.0):
    return models.ToldLuma(len(COLUMNS), 24, 2, 3, scale=SCALE, slope=slope).to(DEVICE)


teacher = told()
teacher.load_state_dict(training.fitted_weights(TEACHER_RUN))
teacher.eval().requires_grad_(False)
print(f"teacher: {models.count_parameters(teacher, True):,} learned parameters, "
      f"student: {models.count_parameters(student(), True):,}")

SOBEL = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]) / 8


def luma(x):
    return metrics.rgb_to_yuv(x)[:, :1]


def edge_strength(y):
    """How strong an edge (N, 1, H, W) luma has at each pixel, the same size (edge repeated)."""
    kernel = torch.stack([SOBEL, SOBEL.T])[:, None].to(y.device, y.dtype)
    return F.pad(F.conv2d(y, kernel).norm(dim=1, keepdim=True), (1, 1, 1, 1), mode="replicate")


# the training set's average edge strength in the truth, which EDGE_STRENGTH is counted against: 2,048 patches
# spread over the split
train_hr = np.load(DATA_DIR / manifest["splits"]["train"]["hr"], mmap_mode="r")
sample = train_hr[np.linspace(0, len(train_hr) - 1, 2048).astype(int), 0]
with torch.no_grad():
    EDGE_SCALE = edge_strength(luma(torch.from_numpy(sample).to(DEVICE).permute(0, 3, 1, 2).float() / 255)).mean().item()
print(f"average luma edge strength in the truth: {EDGE_SCALE:.4f}")


def edge_weighted_loss(prediction, target, alpha=0.84, strength=EDGE_STRENGTH):
    """`ssim_l1_loss` with its L1 weighted by how strong an edge the truth has at each pixel."""
    with torch.no_grad():
        weight = 1 + strength * edge_strength(luma(target)) / EDGE_SCALE
        weight = weight / weight.mean()
    structure = 1 - metrics._ssim(luma(prediction), luma(target)).mean()
    return alpha * structure + (1 - alpha) * (weight * (prediction - target).abs()).mean()


def distill_loss(prediction, target, taught, weight=DISTILL_WEIGHT):
    """`weight` of the loss L1 against the teacher's picture, the rest `ssim_l1_loss` against the truth."""
    return (1 - weight) * metrics.ssim_l1_loss(prediction, target) + weight * F.l1_loss(prediction, taught)

teacher: 0 learned parameters, student: 7,636
average luma edge strength in the truth: 0.0251


## 2. Train

- **The cheap tier**: the luma 32-16 plain at seeds 0 and 1 (seed 0 is notebook 4's run, loaded), and at seed 0
  distilled; then the same three on the leaky body.
- **On notebook 8's told model**: the edge-weighted loss, and a leaky ReLU, each at seed 0. Their plain twin is
  notebook 8's told run at seed 0; the ruler is notebook 7's model at three seeds.

Each run is saved to `checkpoints/distill_runs/` after every epoch.

In [3]:
RUNS = {   # name -> (build, seed, fit settings, run file)
    "luma 32-16": (student, 0, {}, training.CHECKPOINT_DIR / "strategy_runs" / "luma_32-16_seed0.pt"),
    "luma 32-16, seed 1": (student, 1, {}, RUN_DIR / "32-16_seed1.pt"),
    "luma 32-16 distilled": (lambda: Blind(student()), 0, dict(told=TOLD, loss_fn=distill_loss, teacher=teacher),
                             RUN_DIR / "32-16_distilled_seed0.pt"),
    "luma 32-16 leaky": (lambda: student(LEAKY_SLOPE), 0, {}, RUN_DIR / "32-16_leaky_seed0.pt"),
    "luma 32-16 leaky, seed 1": (lambda: student(LEAKY_SLOPE), 1, {}, RUN_DIR / "32-16_leaky_seed1.pt"),
    "luma 32-16 leaky distilled": (lambda: Blind(student(LEAKY_SLOPE)), 0,
                                   dict(told=TOLD, loss_fn=distill_loss, teacher=teacher),
                                   RUN_DIR / "32-16_leaky_distilled_seed0.pt"),
    "told the bitrate and line, edge-weighted": (told, 0, dict(told=TOLD, loss_fn=edge_weighted_loss),
                                                 RUN_DIR / "told_edge_weighted_seed0.pt"),
    "told the bitrate and line, leaky": (lambda: told(LEAKY_SLOPE), 0, dict(told=TOLD), RUN_DIR / "told_leaky_seed0.pt"),
}

In [4]:
runs, select_scores = {}, {}
for name, (build, seed, settings, path) in RUNS.items():
    torch.manual_seed(seed)
    model = build()
    history, best = training.fit(model, path, manifest, seed=seed, verbose=0, **settings)
    runs[name], select_scores[name] = model.eval(), best["score"]
    print(f"{name:<42} best epoch {best['epoch']:3} of {len(history):3}, select SSIM-Y {best['score']:.4f}")

train  140795 patches
select   6982 patches, to choose the best epoch
luma 32-16                                 best epoch 141 of 147, select SSIM-Y 0.8832
train  140795 patches
select   6982 patches, to choose the best epoch
luma 32-16, seed 1                         best epoch 147 of 165, select SSIM-Y 0.8840
train  140795 patches
select   6982 patches, to choose the best epoch
luma 32-16 distilled                       best epoch 101 of 107, select SSIM-Y 0.8844
train  140795 patches
select   6982 patches, to choose the best epoch
luma 32-16 leaky                           best epoch 103 of 115, select SSIM-Y 0.8837
train  140795 patches
select   6982 patches, to choose the best epoch
luma 32-16 leaky, seed 1                   best epoch 103 of 115, select SSIM-Y 0.8840
train  140795 patches
select   6982 patches, to choose the best epoch
luma 32-16 leaky distilled                 best epoch 107 of 113, select SSIM-Y 0.8840
train  140795 patches
select   6982 patches, to choose the

## 3. Scores

Every run on the validation split. Each idea against its plain twin: the two plain seeds' spread is the ruler for the
32-16s, the two leaky seeds' for the leaky ones, and notebook 7's model at three seeds for the told model.

In [5]:
def upscale(name):
    model = runs[name]
    if isinstance(model, models.ToldLuma):
        return lambda lr, condition: model(lr, condition[:, COLUMNS])
    return lambda lr, condition: model(lr)


scores = {}
for name in runs:
    scores[name] = training.val_scores(upscale(name), manifest)
    if name != "luma 32-16":   # notebook 4 recorded it
        training.record(NOTEBOOK, name, scores[name])
told_plain = recorded[8, "told the bitrate and line"]
training.show(training.compare({"bicubic": bicubic, **scores, "told the bitrate and line": told_plain}, manifest,
                               base=bicubic))

mean = lambda s: s["ssim"].mean()
blind_seeds = [s for (notebook, name), s in recorded.items()
               if notebook in (4, 8) and name.startswith("luma 24 + 2 blocks")]
RULERS = {"relu": np.ptp([mean(scores["luma 32-16"]), mean(scores["luma 32-16, seed 1"])]),
          "leaky": np.ptp([mean(scores["luma 32-16 leaky"]), mean(scores["luma 32-16 leaky, seed 1"])]),
          "told": np.ptp([mean(s) for s in blind_seeds])}
print("\nrulers: " + ", ".join(f"{name} {value:.4f}" for name, value in RULERS.items()))


def verdict(a, b, ruler, label):
    word = "luck" if abs(a - b) <= ruler else ("better" if a > b else "worse")
    print(f"{label:<62}{a - b:+.4f} SSIM-Y  {word}")


relu_mean = np.mean([mean(scores["luma 32-16"]), mean(scores["luma 32-16, seed 1"])])
leaky_mean = np.mean([mean(scores["luma 32-16 leaky"]), mean(scores["luma 32-16 leaky, seed 1"])])
verdict(leaky_mean, relu_mean, max(RULERS["relu"], RULERS["leaky"]), "leaky over ReLU, the means of two seeds")
verdict(mean(scores["luma 32-16 distilled"]), relu_mean, RULERS["relu"], "distilled over plain, ReLU")
verdict(mean(scores["luma 32-16 leaky distilled"]), leaky_mean, RULERS["leaky"], "distilled over plain, leaky")
for name in ("told the bitrate and line, edge-weighted", "told the bitrate and line, leaky"):
    verdict(mean(scores[name]), mean(told_plain), RULERS["told"], f"{name.split(', ')[1]} over notebook 8's told model")

                                   model SSIM-Y over bicubic          95% range luma dB dB over chroma dB over   video  screen   photo    x264   nvenc
               told the bitrate and line 0.8905      +0.0384 +0.0361 to +0.0408   30.09   +1.50          +0.04 +0.0219 +0.0570 +0.0237 +0.0398 +0.0370
        told the bitrate and line, leaky 0.8900      +0.0379 +0.0357 to +0.0402   30.09   +1.51          +0.04 +0.0220 +0.0559 +0.0236 +0.0392 +0.0366
told the bitrate and line, edge-weighted 0.8899      +0.0378 +0.0355 to +0.0402   30.07   +1.49          +0.04 +0.0216 +0.0562 +0.0232 +0.0391 +0.0365
                    luma 32-16 distilled 0.8836      +0.0315 +0.0294 to +0.0336   29.79   +1.21          +0.03 +0.0168 +0.0489 +0.0158 +0.0326 +0.0303
                      luma 32-16, seed 1 0.8832      +0.0311 +0.0289 to +0.0333   29.68   +1.10          +0.03 +0.0163 +0.0489 +0.0147 +0.0321 +0.0301
                luma 32-16 leaky, seed 1 0.8828      +0.0307 +0.0286 to +0.0328   29.71   +1.1

### Dead channels

How many channels after each ReLU of a plain 32-16 body are zero on every validation pixel: what a leaky ReLU is
there to prevent.

In [6]:
@torch.no_grad()
def dead_channels(model):
    """Channels after each activation of a plain body that are at most zero on every validation pixel."""
    peaks = None
    for lr, _ in DataLoader(patches.PatchPairs(DATA_DIR, "val", manifest, augment=False), batch_size=256):
        h, batch = lr.to(DEVICE), []
        for layer in model.body:
            h = layer(h)
            if isinstance(layer, (nn.ReLU, nn.LeakyReLU)):
                batch.append(h.amax(dim=(0, 2, 3)))
        peaks = batch if peaks is None else [torch.maximum(a, b) for a, b in zip(peaks, batch)]
    return [int((peak <= 0).sum()) for peak in peaks]


for name in ("luma 32-16", "luma 32-16, seed 1", "luma 32-16 distilled", "luma 32-16 leaky", "luma 32-16 leaky, seed 1"):
    model = runs[name].model if isinstance(runs[name], Blind) else runs[name]
    dead = dead_channels(model)
    print(f"{name:<28} SSIM-Y {mean(scores[name]):.4f}, dead channels: "
          + ", ".join(f"{d} of {w}" for d, w in zip(dead, STUDENT["widths"])))

luma 32-16                   SSIM-Y 0.8822, dead channels: 0 of 32, 5 of 16
luma 32-16, seed 1           SSIM-Y 0.8832, dead channels: 0 of 32, 1 of 16
luma 32-16 distilled         SSIM-Y 0.8836, dead channels: 0 of 32, 1 of 16
luma 32-16 leaky             SSIM-Y 0.8826, dead channels: 0 of 32, 0 of 16
luma 32-16 leaky, seed 1     SSIM-Y 0.8828, dead channels: 0 of 32, 0 of 16


## 4. Export for the browser

The cheap tier: the luma 32-16 leaky at the seed that did better on the select split (never the validation split,
which only scores), at the client's tile and as a whole 1080p frame, in float32 and float16.

In [7]:
seed_name = max(("luma 32-16 leaky", "luma 32-16 leaky, seed 1"), key=select_scores.get)
cheap = runs[seed_name]
print(f"publishing {seed_name}")
webexport.publish(cheap, "upscale_luma_32-16_leaky", cheap.halo, label="luma 32-16 leaky", scale=SCALE, float16=True)
print("compare in a browser: uv run src/upscale/benchmark/main.py")

publishing luma 32-16 leaky, seed 1
upscale_luma_32-16_leaky_tile320x180.onnx              35.0 KB    3.99 ms   Addx1, Concatx1, Convx4, DepthToSpacex1, LeakyRelux2
upscale_luma_32-16_leaky_tile320x180_frame.onnx        35.4 KB   42.66 ms   Addx1, Concatx1, Convx4, DepthToSpacex1, LeakyRelux2
upscale_luma_32-16_leaky_fp16_tile320x180.onnx         18.7 KB   11.75 ms   Addx1, Concatx1, Convx4, DepthToSpacex1, LeakyRelux2
upscale_luma_32-16_leaky_fp16_tile320x180_frame.onnx   18.7 KB  129.75 ms   Addx1, Concatx1, Convx4, DepthToSpacex1, LeakyRelux2
compare in a browser: uv run src/upscale/benchmark/main.py


## 5. What it says

- **A leaky ReLU keeps every channel and narrows the luck, but does not raise the mean.** The plain 32-16 loses 5 of
  its narrow layer's 16 channels at seed 0 and 1 at seed 1, and its two seeds land 0.0010 apart (0.8822 and 0.8832);
  the leaky body loses none, and its seeds land 0.0002 apart (0.8826 and 0.8828). Both pairs average 0.8827. That is
  insurance, not quality: notebook 3's residual 32-16 lost 11 of 16 at one seed and stalled at bicubic, and a leaky
  ReLU costs nothing in the browser (`LeakyRelu` is in the fast set, at the same multiply-adds).
- **Distillation adds nothing measurable.** The distilled ReLU student scores 0.8836, +0.0009 over its body's two
  seeds, inside their 0.0010; it lost 1 channel where its plain twin at the same seed lost 5, which may be all of its
  +0.0014 over that twin. On the leaky body, which loses none, the distilled student lands on the plain seeds
  (0.8827 against 0.8826 and 0.8828).
- **The edge-weighted loss costs**: −0.0006 SSIM-Y against notebook 8's told model at the same seed (the ruler is
  0.0002), and it is behind on screens too (+0.0562 against +0.0570), where the edges it weights are.
- **A leaky ReLU costs the told model**: −0.0005, also behind on screens (+0.0559). Notebook 8's body stays as it is.
- **Published: the luma 32-16 leaky at seed 1**, the better of its two seeds on the select split, at float32 and
  float16: the cheap tier, at the plain 32-16's cost (8,484 multiply-adds an LR pixel) without its lottery.